# # Task 1b: Feature Engineering, Transformation & Imbalance Handling
# ## 10 Academy Week 5&6 – Fraud Detection Challenge
# **Author:** Bereket Feleke  
# **Date:** 28 December 2025  
# 
# **Objective**: 
# - Engineer time-based and velocity features
# - Apply numerical scaling + categorical encoding (ColumnTransformer)
# - Handle class imbalance with SMOTE (only on training data)
# - Save before/after distribution table for interim report (rubric requirement)

In [ ]:
from imblearn.over_sampling import SMOTE
print("SMOTE imported successfully!")

SMOTE imported successfully!


In [18]:
# ## Cell 1: Imports & Setup
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import os

os.makedirs("../reports", exist_ok=True)
print("Task 1b: Imports complete – ready for feature engineering & imbalance handling")

Task 1b: Imports complete – ready for feature engineering & imbalance handling


In [19]:
# Cell 2: Load & Quick Cleaning (Fraud_Data.csv)
fraud_df = pd.read_csv("../data/raw/Fraud_Data.csv")
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])
fraud_df['ip_address'] = fraud_df['ip_address'].astype('int64')

# Quick check
print("Loaded Fraud_Data shape:", fraud_df.shape)
print("Missing values:", fraud_df.isnull().sum().sum())
print("Duplicates:", fraud_df.duplicated().sum())

Loaded Fraud_Data shape: (151112, 11)
Missing values: 0
Duplicates: 0


In [20]:
# Cell 3: Geolocation Mapping (IP → Country)
ip_to_country = pd.read_csv("../data/raw/IpAddress_to_Country.csv")

# Fast vectorized mapping using searchsorted
ip_to_country = ip_to_country.sort_values('lower_bound_ip_address').reset_index(drop=True)

def map_ip_to_country(ip):
    idx = np.searchsorted(ip_to_country['lower_bound_ip_address'], ip, side='right') - 1
    if idx >= 0 and ip <= ip_to_country['upper_bound_ip_address'].iloc[idx]:
        return ip_to_country['country'].iloc[idx]
    return 'Unknown'

fraud_df['country'] = fraud_df['ip_address'].apply(map_ip_to_country)
print("Geolocation mapping complete. Unique countries:", fraud_df['country'].nunique())
print("Unknown IPs:", (fraud_df['country'] == 'Unknown').mean() * 100, "%")

Geolocation mapping complete. Unique countries: 182
Unknown IPs: 14.53623802212928 %


In [21]:
# Cell 4: Feature Engineering
fraud_df['time_since_signup_hours'] = (fraud_df['purchase_time'] - fraud_df['signup_time']).dt.total_seconds() / 3600
fraud_df['hour_of_day'] = fraud_df['purchase_time'].dt.hour
fraud_df['day_of_week'] = fraud_df['purchase_time'].dt.dayofweek
fraud_df['velocity'] = fraud_df['purchase_value'] / (fraud_df['time_since_signup_hours'] + 1)  # avoid div by 0

print("New engineered features:", ['time_since_signup_hours', 'hour_of_day', 'day_of_week', 'velocity'])

# Interpretation:
# - time_since_signup_hours: shorter times often indicate fraud (quick signup-purchase)
# - velocity: high value per hour can signal suspicious behavior

New engineered features: ['time_since_signup_hours', 'hour_of_day', 'day_of_week', 'velocity']


In [22]:
# Cell 5: Define Features & Stratified Split
num_features = ['purchase_value', 'age', 'time_since_signup_hours', 'velocity']
cat_features = ['source', 'browser', 'sex', 'country']  # country added from geolocation

X = fraud_df[num_features + cat_features]
y = fraud_df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (120889, 8) Test shape: (30223, 8)


In [24]:
# Cell 6: Data Transformation Pipeline
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
])

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Force dense arrays (critical for saving & modeling)
X_train_transformed = X_train_transformed.toarray() if hasattr(X_train_transformed, 'toarray') else X_train_transformed
X_test_transformed = X_test_transformed.toarray() if hasattr(X_test_transformed, 'toarray') else X_test_transformed

print("Transformation complete. Train shape:", X_train_transformed.shape)

Transformation complete. Train shape: (120889, 190)


c:\Users\JERUSALEM\Desktop\10 ACA\Fraud-detection-wk56\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [25]:
# Cell 7: Class Imbalance Handling – SMOTE (only on train!)
print("\n=== Class Distribution BEFORE SMOTE (Fraud_Data train) ===")
before = pd.Series(y_train).value_counts(normalize=True).to_frame('Proportion Before SMOTE')
print(before)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_transformed, y_train)

print("\n=== Class Distribution AFTER SMOTE ===")
after = pd.Series(y_train_smote).value_counts(normalize=True).to_frame('Proportion After SMOTE')
print(after)

# Save table for interim report
imbalance_table = pd.concat([before, after], axis=1)
imbalance_table.to_markdown('../reports/task1b_imbalance_table.md')
print("Imbalance table saved to reports/task1b_imbalance_table.md")

# Interpretation:
# - Before SMOTE: ~90.61% legitimate, ~9.39% fraud → severe imbalance
# - After SMOTE: balanced 50/50 → ideal for training (no data leakage to test)


=== Class Distribution BEFORE SMOTE (Fraud_Data train) ===
       Proportion Before SMOTE
class                         
0                     0.906352
1                     0.093648

=== Class Distribution AFTER SMOTE ===
       Proportion After SMOTE
class                        
0                         0.5
1                         0.5
Imbalance table saved to reports/task1b_imbalance_table.md


In [26]:
# Cell 8: Save Processed Data (Dense Format for Modeling)
pd.DataFrame(X_train_smote).to_csv("../data/processed/X_fraud_train_smote.csv", index=False)
pd.Series(y_train_smote).to_csv("../data/processed/y_fraud_train_smote.csv", index=False)
pd.DataFrame(X_test_transformed).to_csv("../data/processed/X_fraud_test.csv", index=False)
pd.Series(y_test).to_csv("../data/processed/y_fraud_test.csv", index=False)

print("Processed files saved as dense numeric arrays (ready for modeling)!")

Processed files saved as dense numeric arrays (ready for modeling)!


In [27]:
# Cell 9: Conclusion & Next Steps
print("\n=== Task 1b Complete ===")
print("- Geolocation mapping completed")
print("- Features engineered: time_since_signup_hours, hour_of_day, day_of_week, velocity")
print("- Transformation pipeline applied (scaling + encoding)")
print("- SMOTE applied to training data only → balanced classes")
print("- Imbalance table saved for interim report")
print("- Processed files saved as dense (no sparse string issue for modeling)")

print("\nNext: Modeling (Task 2) – load X_fraud_train_smote.csv & y_fraud_train_smote.csv")


=== Task 1b Complete ===
- Geolocation mapping completed
- Features engineered: time_since_signup_hours, hour_of_day, day_of_week, velocity
- Transformation pipeline applied (scaling + encoding)
- SMOTE applied to training data only → balanced classes
- Imbalance table saved for interim report
- Processed files saved as dense (no sparse string issue for modeling)

Next: Modeling (Task 2) – load X_fraud_train_smote.csv & y_fraud_train_smote.csv
